# 05 - Evaluation and SHAP

Evaluating the final thesis-facing **SE-HC / SIGMA_FINAL** model. OOF predictions are used for detailed validation because Kaggle test labels are not public. The Kaggle private leaderboard score is reported as an external benchmark, while OOF metrics remain the main controlled validation evidence. SHAP is used for explainability.

Final thesis terminology:

- SPC = Stacked Prediction Candidate = `fp_final`.
- SE-HC = `SIGMA_FINAL`.
- Formula: `SE-HC = 0.5 * SPC + 0.5 * V15 Multi-Seed LightGBM`.
- Final SE-HC OOF ROC-AUC = `0.801688`.


## Section 2 - Load final SE-HC results

SE-HC refers to `SIGMA_FINAL` in the implementation. This section loads or references lightweight report files for GitHub/demo and the local-only final OOF artifact when available.


In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPORTS = ROOT / 'reports'
RESULTS = REPORTS / 'results'
FIGURES = REPORTS / 'figures'
OOF_DIR = ROOT / 'outputs' / 'oof_predictions'
SHAP_DIR = ROOT / 'outputs' / 'shap_output'

paths = pd.DataFrame([
    {'artifact':'metrics_SE_HC.json','path':'reports/results/metrics_SE_HC.json','exists':(RESULTS / 'metrics_SE_HC.json').exists()},
    {'artifact':'deciles_SE_HC.csv','path':'reports/results/deciles_SE_HC.csv','exists':(RESULTS / 'deciles_SE_HC.csv').exists()},
    {'artifact':'final_v15_baseline_comparison.csv','path':'reports/results/final_v15_baseline_comparison.csv','exists':(RESULTS / 'final_v15_baseline_comparison.csv').exists()},
    {'artifact':'dashboard_SE_HC.png','path':'reports/figures/dashboard_SE_HC.png','exists':(FIGURES / 'dashboard_SE_HC.png').exists()},
    {'artifact':'oof_SIGMA_FINAL.parquet','path':'outputs/oof_predictions/oof_SIGMA_FINAL.parquet','exists':(OOF_DIR / 'oof_SIGMA_FINAL.parquet').exists()},
])
paths


                         artifact                                              path  exists
               metrics_SE_HC.json                reports/results/metrics_SE_HC.json    True
                deciles_SE_HC.csv                 reports/results/deciles_SE_HC.csv    True
final_v15_baseline_comparison.csv reports/results/final_v15_baseline_comparison.csv    True
              dashboard_SE_HC.png               reports/figures/dashboard_SE_HC.png    True
          oof_SIGMA_FINAL.parquet   outputs/oof_predictions/oof_SIGMA_FINAL.parquet    True

## Section 3 - OOF evaluation metrics

ROC-AUC evaluates ranking quality. KS measures separation between default and non-default distributions. PR-AUC is useful for imbalanced default detection. Lift@10% measures business usefulness in the riskiest applicant segment. Brier and ECE are reported as secondary probability-quality indicators, not the main thesis focus.


In [ ]:

metrics = json.loads((RESULTS / 'metrics_SE_HC.json').read_text(encoding='utf-8'))
metrics_table = pd.DataFrame([
    {'Metric':'ROC-AUC','Value':metrics['roc_auc'],'Interpretation':'Ranking quality for default vs non-default applicants'},
    {'Metric':'Gini','Value':metrics['gini'],'Interpretation':'Credit-risk ranking transform of ROC-AUC'},
    {'Metric':'KS','Value':metrics['ks'],'Interpretation':'Separation between default and non-default score distributions'},
    {'Metric':'PR-AUC','Value':metrics['pr_auc'],'Interpretation':'Useful under class imbalance'},
    {'Metric':'Lift@10%','Value':metrics['lift_10pct'],'Interpretation':'Business usefulness in the riskiest 10% segment'},
    {'Metric':'Lift@20%','Value':metrics['lift_20pct'],'Interpretation':'Business usefulness in the riskiest 20% segment'},
    {'Metric':'Brier','Value':metrics['brier'],'Interpretation':'Secondary probability-quality indicator'},
    {'Metric':'ECE','Value':metrics['ece'],'Interpretation':'Secondary calibration indicator'},
])
metrics_table


  Metric    Value                                                 Interpretation
 ROC-AUC 0.801688          Ranking quality for default vs non-default applicants
    Gini 0.603377                       Credit-risk ranking transform of ROC-AUC
      KS 0.457192 Separation between default and non-default score distributions
  PR-AUC 0.300543                                   Useful under class imbalance
Lift@10% 3.944501                Business usefulness in the riskiest 10% segment
Lift@20% 2.955709                Business usefulness in the riskiest 20% segment
   Brier 0.064796                        Secondary probability-quality indicator
     ECE 0.001940                                Secondary calibration indicator

## Kaggle external benchmark

The final SE-HC submission achieved a private leaderboard score of **0.79939**, placing it within the top tier of the competition leaderboard. In the leaderboard at the time of 26/5/2026, this score is approximately rank around top 1%. Since leaderboard positions may shift with late submissions, the private score is reported as an external benchmark rather than the sole model-selection criterion.


## Threshold analysis

A threshold of `0.5` is not suitable for imbalanced credit risk. Lower operating thresholds are needed when the cost of missing a default is higher than incorrectly flagging a non-default applicant. Threshold choice is business-dependent.

The operating threshold below is the Youden/KS-style threshold saved from the local evaluation artifacts.


In [ ]:

threshold = json.loads((ROOT / 'outputs' / 'others' / 'classification_metrics_at_threshold.json').read_text(encoding='utf-8'))
threshold_table = pd.DataFrame([{
    'threshold_t*': threshold['optimal_threshold'],
    'precision': threshold['precision'],
    'recall': threshold['recall'],
    'f1': threshold['f1'],
    'accuracy': threshold['accuracy'],
    'specificity': threshold['specificity'],
    'TN': threshold['confusion_matrix']['TN'],
    'FP': threshold['confusion_matrix']['FP'],
    'FN': threshold['confusion_matrix']['FN'],
    'TP': threshold['confusion_matrix']['TP'],
}])
threshold_table


 threshold_t*  precision   recall       f1  accuracy  specificity     TN    FP   FN    TP
     0.080135   0.188365 0.735509 0.299920  0.722800     0.721684 204007 78675 6566 18259

Cached confusion-matrix figure if available:

![Confusion matrix](../outputs/others/confusion_matrix.png)


## Decile and lift analysis

The top risk decile should have a much higher default rate than the portfolio average. This supports using the model for risk ranking and prioritization. Decile analysis is easier to interpret for credit-risk business users than only reporting ROC-AUC.


In [ ]:

deciles = pd.read_csv(RESULTS / 'deciles_SE_HC.csv')
deciles


 bucket     n  defaults  default_rate     lift  cum_capture
     10 30751       218      0.007100 0.087800     0.008800
      9 30751       420      0.013700 0.169200     0.025700
      8 30750       589      0.019200 0.237300     0.049400
      7 30751       826      0.026900 0.332700     0.082700
      6 30751      1105      0.035900 0.445100     0.127200
      5 30750      1551      0.050400 0.624800     0.189700
      4 30751      2262      0.073600 0.911200     0.280800
      3 30750      3179      0.103400 1.280600     0.408900
      2 30751      4883      0.158800 1.966900     0.605600
      1 30751      9792      0.318400 3.944400     1.000000

Final dashboard and demo decile figure:

![SE-HC dashboard](../reports/figures/dashboard_SE_HC.png)

![Decile risk segmentation](../reports/figures/decile_risk_segmentation.png)


## Baseline comparison

This section uses the corrected V15 baseline comparison computed from canonical OOF files. The lightweight CSV at `reports/results/final_v15_baseline_comparison.csv` is referenced for repository readability, but the table below avoids the old mismatched baseline table by recomputing from canonical OOF predictions when available.

Logistic Regression and MLP are baseline models. LightGBM, XGBoost, and CatBoost are stronger tree-based baselines. SE-HC improves over individual baselines by combining complementary prediction signals.


In [ ]:

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, f1_score, precision_score, recall_score, roc_curve

ID_COL = 'SK_ID_CURR'
TARGET = 'TARGET'

def ks_stat(y_true, y_pred):
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    return float(np.max(tpr - fpr))

def lift_at_k(y_true, y_pred, k=0.1):
    n_top = int(len(y_true) * k)
    top_idx = np.argsort(y_pred)[::-1][:n_top]
    return float(y_true[top_idx].mean() / y_true.mean())

def ece_metric(y_true, y_pred, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_pred >= bins[i]) & (y_pred < bins[i + 1])
        if mask.sum() > 0:
            ece += abs(y_pred[mask].mean() - y_true[mask].mean()) * mask.sum() / len(y_true)
    return float(ece)

baseline_files = {
    'Logistic Regression': ('oof_lr_v15_baseline.parquet', 'oof_lr'),
    'MLP': ('oof_mlp_v15.parquet', 'oof_mlp'),
    'LightGBM single seed': ('oof_lgb_v15_single_seed.parquet', 'oof_lgb'),
    'LightGBM multi-seed': ('oof_lgb_v15_multiseed.parquet', 'oof_lgb'),
    'XGBoost': ('oof_xgb_v15.parquet', 'oof_xgb'),
    'CatBoost': ('oof_cb_v15.parquet', 'oof_cb'),
    'SE-HC / SIGMA_FINAL': ('oof_SIGMA_FINAL.parquet', 'oof_final'),
}

rows = []
for model, (filename, pred_col) in baseline_files.items():
    path = OOF_DIR / filename
    if not path.exists():
        continue
    df = pd.read_parquet(path)
    y = df[TARGET].astype(int).to_numpy()
    pred = df[pred_col].astype(float).to_numpy()
    fpr, tpr, thresholds = roc_curve(y, pred)
    t_star = thresholds[np.argmax(tpr - fpr)]
    binary = (pred >= t_star).astype(int)
    auc = roc_auc_score(y, pred)
    rows.append({
        'Model': model,
        'ROC-AUC': auc,
        'Gini': 2 * auc - 1,
        'KS': ks_stat(y, pred),
        'PR-AUC': average_precision_score(y, pred),
        'Lift@10%': lift_at_k(y, pred),
        'F1 (t*)': f1_score(y, binary),
        'Precision': precision_score(y, binary),
        'Recall': recall_score(y, binary),
        'Brier': brier_score_loss(y, pred),
        'ECE': ece_metric(y, pred),
        't*': t_star,
    })

corrected_v15_comparison = pd.DataFrame(rows)
corrected_v15_comparison


               Model  ROC-AUC     Gini       KS   PR-AUC  Lift@10%  F1 (t*)  Precision   Recall    Brier      ECE       t*
 Logistic Regression 0.782546 0.565092 0.424540 0.267877  3.658492 0.286685   0.179690 0.708640 0.070328 0.061353 0.155053
                 MLP 0.725532 0.451063 0.337656 0.204363  3.043372 0.247828   0.152609 0.659013 0.075444 0.043269 0.045007
LightGBM single seed 0.798504 0.597009 0.452538 0.295101  3.898981 0.298229   0.187321 0.731078 0.065093 0.002370 0.076887
 LightGBM multi-seed 0.799749 0.599497 0.453514 0.297331  3.898175 0.292224   0.181390 0.751259 0.064972 0.002716 0.072577
             XGBoost 0.798955 0.597910 0.452732 0.296833  3.912274 0.302721   0.191730 0.718872 0.065011 0.002048 0.081166
            CatBoost 0.798170 0.596340 0.452687 0.295139  3.907440 0.293298   0.182520 0.746183 0.065092 0.003858 0.073554
 SE-HC / SIGMA_FINAL 0.801688 0.603377 0.457192 0.300543  3.944501 0.299920   0.188365 0.735509 0.064796 0.002831 0.080135

## SHAP explainability

Since the final SE-HC model is a blend of candidate prediction vectors, SHAP is reported for a representative tree-based component of the pipeline. The explanations should be interpreted as evidence of the main predictive patterns used by the ensemble components, not as an exact decomposition of the final blended SE-HC probability.

Cached SHAP artifacts are used here. Full SHAP recomputation is local-only and should not be run during the demo unless the cached model/data are available.


In [ ]:

shap_importance = pd.read_csv(SHAP_DIR / 'shap_importance.csv')
shap_categories = pd.read_csv(SHAP_DIR / 'shap_by_category.csv')
shap_cases = json.loads((SHAP_DIR / 'shap_case_studies.json').read_text(encoding='utf-8'))
shap_interactions = pd.read_csv(SHAP_DIR / 'shap_top_interactions.csv')

shap_importance.head(15)


               feature  mean_abs_shap  mean_shap  std_shap
     TARGET_NN500_MEAN       0.240272  -0.004030  0.288019
        BUR_SCORE_MEAN       0.172819  -0.002359  0.190452
       PREV_SCORE_MEAN       0.131058   0.001370  0.151084
PREDICTED_EXT_SOURCE_1       0.127257  -0.008717  0.144893
 EXT1_PREDICTED_x_EXT2       0.099866   0.009787  0.112754
              EXT_MEAN       0.092956  -0.000497  0.105974
     ORGANIZATION_TYPE       0.090426  -0.000315  0.122759
           AMT_ANNUITY       0.083956   0.000181  0.096032
 EXT3_PREDICTED_x_EXT2       0.071032  -0.003046  0.083604
       AMT_GOODS_PRICE       0.068174   0.003200  0.077668
           CODE_GENDER       0.067546  -0.002433  0.074179
           OWN_CAR_AGE       0.061452  -0.000472  0.073461
    NAME_FAMILY_STATUS       0.060646  -0.001271  0.066362
            AMT_CREDIT       0.058104  -0.001222  0.071783
            DAYS_BIRTH       0.053498   0.004960  0.084609

Cached global SHAP figures:

![SHAP bar top 20](../reports/figures/shap_bar_top20.png)

![SHAP beeswarm top 20](../reports/figures/shap_beeswarm_top20.png)


In [ ]:

shap_categories.sort_values('Total_SHAP', ascending=False)


                   Category  N_features  Total_SHAP  Mean_SHAP  Max_SHAP
      External Credit Score          10    0.504155   0.050415  0.127257
       Sub-Model Risk Score          13    0.487856   0.037527  0.172819
Demographics and Employment           7    0.358090   0.051156  0.090426
     Application Financials           7    0.283106   0.040444  0.083956
  Neighbourhood Risk Signal           2    0.266510   0.133255  0.240272
      Credit Bureau History           0    0.000000        NaN       NaN
       Instalment Behaviour           0    0.000000        NaN       NaN
       Previous Application           0    0.000000        NaN       NaN
          POS / Credit Card           0    0.000000        NaN       NaN
            Document / Misc           0    0.000000        NaN       NaN

Cached category-level SHAP figure:

![SHAP category importance](../reports/figures/shap_category_importance.png)


In [ ]:

pd.DataFrame(shap_cases).T


      idx      pred true     sk_id                                                       desc
TP   57.0  0.602292  1.0  129426.0     True Positive (correctly identified high-risk default)
TN    1.0  0.007724  0.0  405321.0         True Negative (correctly identified safe customer)
FP   22.0  0.404956  0.0  368049.0  False Positive (wrongly flagged - rejected good customer)
FN  223.0  0.040006  1.0  356366.0    False Negative (missed default - approved bad customer)

Cached local SHAP example figure:

![Local SHAP example](../reports/figures/local_shap.png)

Additional cached waterfall plots are stored locally under `evaluation/figures_shap/`.


In [ ]:

shap_interactions.head(10)


            feature_1             feature_2  interaction_strength
           AMT_CREDIT       AMT_GOODS_PRICE              0.017541
   NAME_CONTRACT_TYPE           AMT_ANNUITY              0.016717
    TARGET_NN500_MEAN        BUR_SCORE_MEAN              0.013228
    TARGET_NN500_MEAN       PREV_SCORE_MEAN              0.012410
    TARGET_NN500_MEAN EXT1_PREDICTED_x_EXT2              0.009204
          AMT_ANNUITY       AMT_GOODS_PRICE              0.009178
EXT1_PREDICTED_x_EXT2        BUR_SCORE_MEAN              0.007116
      PREV_SCORE_MEAN        BUR_SCORE_MEAN              0.006768
             EXT_MEAN       PREV_SCORE_MEAN              0.006537
    ORGANIZATION_TYPE       PREV_SCORE_MEAN              0.006451

Cached dependence and interaction figures are available locally under `evaluation/figures_shap/`, including dependence plots for `TARGET_NN500_MEAN`, `BUR_SCORE_MEAN`, `PREV_SCORE_MEAN`, `PREDICTED_EXT_SOURCE_1`, and `EXT1_PREDICTED_x_EXT2`.


## Business interpretation

High external risk scores reduce predicted risk. Younger age, shorter employment history, high credit burden, weak bureau history, past delinquency, and high utilization increase predicted risk.

Bureau and previous-application behavioral features provide stronger predictive power than many raw applicant attributes. This is consistent with the model relying heavily on repayment history, external credit behavior, and engineered row-level risk-score features rather than only static demographic fields.


In [ ]:

local_only_artifacts = pd.DataFrame([
    {'artifact':'Full SE-HC OOF predictions','path':'outputs/oof_predictions/oof_SIGMA_FINAL.parquet'},
    {'artifact':'Full SHAP values','path':'outputs/shap_output/shap_values.npy'},
    {'artifact':'SHAP sample index','path':'outputs/shap_output/shap_sample_idx.npy'},
    {'artifact':'Cached SHAP model','path':'outputs/shap_output/lgbm_shap_model.pkl'},
])
local_only_artifacts['exists_locally'] = local_only_artifacts['path'].map(lambda p: (ROOT / p).exists())
local_only_artifacts


                  artifact                                            path  exists_locally
Full SE-HC OOF predictions outputs/oof_predictions/oof_SIGMA_FINAL.parquet            True
          Full SHAP values             outputs/shap_output/shap_values.npy            True
         SHAP sample index         outputs/shap_output/shap_sample_idx.npy            True
         Cached SHAP model         outputs/shap_output/lgbm_shap_model.pkl            True